# Evaluación de modelos

Qué miramos para saber si un modelo funciona bien (con desbalance de clases):
1. comparar corridas (MLflow),
2. curvas train vs val (overfitting),
3. métricas en **val y test** (macro-F1, balanced acc, F1 por clase),
4. matriz de confusión (qué se confunde con qué),
5. vista clínica **maligno vs benigno** (el error grave: un maligno clasificado como benigno).

Requiere haber entrenado al menos una vez (existe `models/model.pt` y `mlruns/`).

In [ ]:
import os
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"

import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
from mlflow.tracking import MlflowClient
from sklearn.metrics import classification_report, confusion_matrix, recall_score
from torch.utils.data import DataLoader

from src.data.dataset_utils import CLASSES, MALIGNANT
from src.models.architectures import build_model
from src.models.dataset import MultimodalDataset
from src.models.train import get_device, MLFLOW_URI

mlflow.set_tracking_uri(MLFLOW_URI)
MODELS_DIR = ROOT / "models"
device = get_device()
print("device:", device, "| mlflow:", MLFLOW_URI)

## 1. Comparar corridas (MLflow)

In [ ]:
runs = mlflow.search_runs(experiment_names=["skin-lesion"])
cols = ["run_id", "metrics.best_val_macro_f1", "params.architecture",
        "params.unfreeze_blocks", "params.lr", "params.epochs"]
cols = [c for c in cols if c in runs.columns]
runs[cols].sort_values("metrics.best_val_macro_f1", ascending=False)

## 2. Curvas de la mejor corrida (overfitting)

In [ ]:
best_run_id = runs.sort_values("metrics.best_val_macro_f1", ascending=False).iloc[0]["run_id"]
client = MlflowClient()

def history(metric):
    h = client.get_metric_history(best_run_id, metric)
    h = sorted(h, key=lambda m: m.step)
    return [m.step for m in h], [m.value for m in h]

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for m in ["train_loss", "val_loss"]:
    s, v = history(m); ax[0].plot(s, v, marker="o", label=m)
ax[0].set_title("Pérdida"); ax[0].set_xlabel("época"); ax[0].legend()
s, v = history("val_macro_f1"); ax[1].plot(s, v, marker="o", color="green")
ax[1].set_title("val macro-F1"); ax[1].set_xlabel("época")
plt.show()

## 3. Cargar el mejor modelo guardado (`models/model.pt`)

In [ ]:
ckpt = torch.load(MODELS_DIR / "model.pt", map_location=device)
model = build_model(ckpt["architecture"], **ckpt["arch_kwargs"]).to(device)
model.load_state_dict(ckpt["state_dict"])
model.eval()
classes = ckpt["classes"]
print("arquitectura:", ckpt["architecture"], "| clases:", classes)

## 4. Métricas en val y test

Nota: el **test** es el juez final y se mira con cuidado. Comparar val vs test dice si el modelo
generaliza o si nos "casamos" con val.

In [ ]:
@torch.no_grad()
def evaluate(split):
    loader = DataLoader(MultimodalDataset(split), batch_size=32)
    preds, targets = [], []
    for image, tabular, label in loader:
        logits = model(image.to(device), tabular.to(device))
        preds.append(logits.argmax(1).cpu().numpy())
        targets.append(label.numpy())
    return np.concatenate(targets), np.concatenate(preds)

results = {split: evaluate(split) for split in ["val", "test"]}

for split, (y_true, y_pred) in results.items():
    print(f"===== {split.upper()} =====")
    print(classification_report(y_true, y_pred, target_names=classes, digits=3, zero_division=0))

## 5. Matriz de confusión (val y test)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (split, (y_true, y_pred)) in zip(axes, results.items()):
    cm = confusion_matrix(y_true, y_pred, labels=range(len(classes)))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=classes,
                yticklabels=classes, ax=ax)
    ax.set_title(f"Confusión — {split}"); ax.set_xlabel("predicho"); ax.set_ylabel("real")
plt.tight_layout(); plt.show()

## 6. Vista clínica: maligno vs benigno

El error más costoso es predecir **benigno** cuando en realidad es **maligno** (un cáncer que se
escapa). Miramos el **recall de la clase maligna**: de todos los malignos reales, cuántos detecta.

In [ ]:
mal_idx = [i for i, c in enumerate(classes) if c in MALIGNANT]

for split, (y_true, y_pred) in results.items():
    t = np.isin(y_true, mal_idx)
    p = np.isin(y_pred, mal_idx)
    cm = confusion_matrix(t, p, labels=[False, True])
    print(f"{split}: recall maligno = {recall_score(t, p):.3f}")
    print(pd.DataFrame(cm, index=["real benigno", "real maligno"],
                       columns=["pred benigno", "pred maligno"]))
    print()

## 7. Conclusiones / qué priorizar

- Si el **macro-F1 de test** cae mucho respecto a val → el modelo no generaliza (revisar overfitting).
- Mirar **qué clase tiene el F1 más bajo** (probablemente MEL/SCC) → ahí atacar con fine-tuning,
  augmentation u oversampling.
- En la matriz de confusión, ver **con qué se confunde** cada maligno.
- El **recall maligno** es la métrica clínica clave: cuántos cánceres se escapan como benignos.